# Prediksi Keparahan Kemoterapi pada Pasien Anak (STANDALONE)

Notebook ini berisi alur kerja lengkap dan mandiri tanpa bergantung pada file modul eksternal. Semua fungsi untuk pembersihan data, training model, hingga simulasi terintegrasi langsung di dalam sel-sel notebook ini.

**Persiapan di Google Colab**:
1. Pastikan Anda mengunggah file `Master Tabel-2.xlsx` ke *session storage* Colab (pada panel file di kiri).
2. Klik **Runtime > Run All** untuk menjalankan seluruh alur secara otomatis.

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

try:
    from xgboost import XGBClassifier
    xgb_model = XGBClassifier(objective='multi:softprob', random_state=42, eval_metric='mlogloss')
    xgb_name = 'XGBoost'
except (ImportError, Exception):
    # Fallback ke GradientBoosting jika XGBoost gagal diimport
    from sklearn.ensemble import GradientBoostingClassifier
    xgb_model = GradientBoostingClassifier(random_state=42)
    xgb_name = 'XGBoost'

## 2. Pembersihan Data
Membaca file Excel mentah `Master Tabel-2.xlsx` dan memprosesnya hingga siap digunakan untuk pemodelan.

In [ ]:
def bersihkan_dataset_kemo(raw_path="Master Tabel-2.xlsx"):
    print("Memulai proses pembersihan data...")
    clean_path = "dataset_clean.csv"
    
    if not os.path.exists(raw_path):
        raise FileNotFoundError(f"File {raw_path} tidak ditemukan! Harap upload file tersebut terlebih dahulu.")

    # Membaca excel tanpa header agar index kolom sesuai dengan angka 0-35
    df_raw = pd.read_excel(raw_path, sheet_name=0, header=None)

    # Mengambil data dari baris indeks 5 sebanyak 100 baris (pasien valid)
    df = df_raw.iloc[5:105].copy()
    
    # 1. Pilih kolom yang relevan
    kolom_relevan = [0, 4, 5, 6, 7, 8, 9, 10, 11] + list(range(12, 19)) + [25] + list(range(26, 31)) + [35]
    df = df[kolom_relevan].copy()
    
    # Memberi nama kolom untuk memudahkan
    nama_kolom = [
        'no', 'nama_anak', 'usia_str', 'jenis_kelamin', 'diagnosis', 'lama_terdiagnosis', 
        'siklus_kemoterapi_str', 'protokol_kemo', 'riwayat_ranap',
        'mual', 'muntah', 'fatigue', 'diare', 'konstipasi', 'mukositis', 'nyeri',
        'dukungan_keluarga', 
        'hb', 'leukosit', 'neutrofil', 'trombosit', 'suhu',
        'target_severity_str'
    ]
    df.columns = nama_kolom
    
    # 2. Parsing format usia
    def parse_usia_tahun(val):
        val = str(val).lower().replace(',', '.')
        match_thn = re.search(r'(\d+(?:\.\d+)?)\s*tahun', val)
        match_bln = re.search(r'(\d+(?:\.\d+)?)\s*bulan', val)
        
        if match_thn or match_bln:
            thn = float(match_thn.group(1)) if match_thn else 0.0
            bln = float(match_bln.group(1)) if match_bln else 0.0
            return thn + (bln / 12.0)
            
        nums = re.findall(r'\d+(?:\.\d+)?', val)
        if nums:
            return float(nums[0])
        return np.nan
        
    df['usia_tahun'] = df['usia_str'].apply(parse_usia_tahun).astype('float64')

    # 3. Ekstraksi angka siklus kemo
    def parse_siklus(val):
        val = str(val).lower().strip()
        match = re.search(r'\d+', val)
        if match:
            return int(match.group(0))
        return np.nan
    df['siklus_ke'] = df['siklus_kemoterapi_str'].apply(parse_siklus).astype('float64')

    # 4. Mapping ordinal CTCAE
    gejala_cols = ['mual', 'muntah', 'fatigue', 'diare', 'konstipasi', 'mukositis', 'nyeri']
    map_gejala = {
        'tidak ada': 0, 'ringan': 1, 'sedang, berat': 2, 'sedang': 2, 'berat': 3
    }
    for col in gejala_cols:
        def parse_gejala(val):
            val = str(val).lower().strip()
            if val in map_gejala: return map_gejala[val]
            for k, v in map_gejala.items():
                if k in val: return v
            return np.nan
        df[col] = df[col].apply(parse_gejala)

    # 5. Mapping dukungan keluarga & jenis kelamin
    def parse_dukungan(val):
        val = str(val).lower()
        if 'rendah' in val: return 0
        elif 'sedang' in val: return 1
        elif 'tinggi' in val: return 2
        return np.nan
    df['dukungan_keluarga'] = df['dukungan_keluarga'].apply(parse_dukungan)

    def parse_jk(val):
        val = str(val).lower()
        if 'laki' in val: return 0
        elif 'perempuan' in val: return 1
        return np.nan
    df['jenis_kelamin'] = df['jenis_kelamin'].apply(parse_jk)

    # 6. Ekstrak data lab kontinu
    lab_cols = ['hb', 'leukosit', 'neutrofil', 'trombosit', 'suhu']
    for col in lab_cols:
        df[col] = df[col].astype(str).str.replace(',', '.')
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # 7. Mapping target keparahan
    def parse_target(val):
        val = str(val).lower()
        if 'ringan' in val: return 0
        elif 'sedang' in val: return 1
        elif 'berat' in val: return 2
        return np.nan
    df['target_severity'] = df['target_severity_str'].apply(parse_target)
    
    # Hapus baris target NaN
    df = df.dropna(subset=['target_severity'])
    
    # Hapus kolom string asli
    kolom_hapus = ['usia_str', 'siklus_kemoterapi_str', 'target_severity_str']
    df = df.drop(columns=kolom_hapus)

    # 8. IMPUTASI PENUH AGAR BEBAS NaN
    fitur_numerik = ['usia_tahun', 'siklus_ke', 'hb', 'leukosit', 'neutrofil', 'trombosit', 'suhu']
    fitur_kategorik = ['jenis_kelamin', 'mual', 'muntah', 'fatigue', 'diare', 'konstipasi', 'mukositis', 'nyeri', 'dukungan_keluarga']
    
    for col in fitur_numerik:
        df[col] = df[col].fillna(df[col].median())
        
    for col in fitur_kategorik:
        df[col] = df[col].fillna(df[col].mode()[0])
        
    df['siklus_ke'] = df['siklus_ke'].astype(int)
    for col in fitur_kategorik:
        df[col] = df[col].astype(int)

    df.to_csv(clean_path, index=False)
    print(f"Data berhasil dibersihkan dan disimpan di: {clean_path}")
    
    return df, fitur_numerik, fitur_kategorik

# Jalankan pembersihan (pastikan Master Tabel-2.xlsx sudah di-upload)
df_clean, fitur_numerik, fitur_kategorik = bersihkan_dataset_kemo("Master Tabel-2.xlsx")
display(df_clean.head())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df_clean, x='target_severity', palette='viridis')
plt.title('Distribusi Target Keparahan (0=Ringan, 1=Sedang, 2=Berat)')
plt.xlabel('Target Severity')
plt.ylabel('Jumlah Pasien')
plt.show()

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.histplot(df_clean['usia_tahun'], kde=True, bins=15, color='blue')
plt.title('Distribusi Usia (Tahun)')

plt.subplot(1, 2, 2)
sns.histplot(df_clean['siklus_ke'], kde=True, bins=10, color='orange')
plt.title('Distribusi Siklus Kemoterapi')

plt.tight_layout()
plt.show()

## 4. Evaluasi 5-Fold Cross Validation

In [ ]:
def jalankan_validasi_dan_dapatkan_model(df, f_num, f_cat):
    X = df[f_num + f_cat]
    y = df['target_severity'].astype(int)
    
    # Preprocessing Pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), f_num),
            ('cat', SimpleImputer(strategy='most_frequent'), f_cat)
        ]
    )
    
    models = {
        'Logistic Regression': LogisticRegression(class_weight='balanced', multi_class='ovr', random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=42),
        xgb_name: xgb_model
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    hasil_evaluasi = []
    
    for nama_model, model in models.items():
        print(f"Melatih model: {nama_model}...")
        metrics_fold = {'acc': [], 'prec': [], 'rec': [], 'f1': [], 'roc_auc': []}
        
        for train_idx, val_idx in skf.split(X, y):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            pipeline = Pipeline([('prep', preprocessor), ('clf', model)])
            
            if nama_model == 'XGBoost' and 'GradientBoosting' not in str(type(model)):
                s_weight = compute_sample_weight('balanced', y_train)
                pipeline.fit(X_train, y_train, clf__sample_weight=s_weight)
            else:
                pipeline.fit(X_train, y_train)
                
            y_pred = pipeline.predict(X_val)
            y_proba = pipeline.predict_proba(X_val)
            
            metrics_fold['acc'].append(accuracy_score(y_val, y_pred))
            metrics_fold['prec'].append(precision_score(y_val, y_pred, average='macro', zero_division=0))
            metrics_fold['rec'].append(recall_score(y_val, y_pred, average='macro', zero_division=0))
            metrics_fold['f1'].append(f1_score(y_val, y_pred, average='macro', zero_division=0))
            
            try:
                auc = roc_auc_score(y_val, y_proba, average='macro', multi_class='ovr')
            except ValueError:
                auc = np.nan
            metrics_fold['roc_auc'].append(auc)
            
        mean_f1 = np.mean(metrics_fold['f1'])
        hasil_evaluasi.append({
            'Model': nama_model,
            'Accuracy': f"{np.mean(metrics_fold['acc']):.3f} ± {np.std(metrics_fold['acc']):.3f}",
            'Precision (Macro)': f"{np.mean(metrics_fold['prec']):.3f} ± {np.std(metrics_fold['prec']):.3f}",
            'Recall (Macro)': f"{np.mean(metrics_fold['rec']):.3f} ± {np.std(metrics_fold['rec']):.3f}",
            'F1-Score (Macro)': f"{mean_f1:.3f} ± {np.std(metrics_fold['f1']):.3f}",
            'ROC-AUC (Macro)': f"{np.nanmean(metrics_fold['roc_auc']):.3f} ± {np.nanstd(metrics_fold['roc_auc']):.3f}",
            'mean_f1_val': mean_f1
        })
        
    df_hasil = pd.DataFrame(hasil_evaluasi)
    print("\nHasil Evaluasi 5-Fold CV:")
    display(df_hasil.drop(columns=['mean_f1_val']))
    
    model_terbaik_nama = df_hasil.loc[df_hasil['mean_f1_val'].idxmax()]['Model']
    print(f"\nModel terbaik berdasarkan F1-Score: {model_terbaik_nama}. Melakukan refit pada seluruh data...")
    
    model_terbaik = models[model_terbaik_nama]
    pipeline_terbaik = Pipeline([('prep', preprocessor), ('clf', model_terbaik)])
    
    if model_terbaik_nama == 'XGBoost' and 'GradientBoosting' not in str(type(model_terbaik)):
        s_weight = compute_sample_weight('balanced', y)
        pipeline_terbaik.fit(X, y, clf__sample_weight=s_weight)
    else:
        pipeline_terbaik.fit(X, y)
        
    return pipeline_terbaik, model_terbaik_nama

model_terbaik, nama_model_terbaik = jalankan_validasi_dan_dapatkan_model(df_clean, fitur_numerik, fitur_kategorik)

## 5. Simulasi Prediksi Pasien Baru

In [ ]:
# Simulasi data pasien baru
data_pasien_baru = {
    'usia_tahun': [8.5],
    'siklus_ke': [3],
    'hb': [10.2],
    'leukosit': [4500],
    'neutrofil': [1500],
    'trombosit': [120000],
    'suhu': [37.8],
    'jenis_kelamin': [1], # 1 = Perempuan
    'mual': [2],         # 2 = Sedang/Berat
    'muntah': [1],       # 1 = Ringan
    'fatigue': [1],
    'diare': [0],
    'konstipasi': [0],
    'mukositis': [1],
    'nyeri': [2],
    'dukungan_keluarga': [2] # 2 = Tinggi
}

df_pasien = pd.DataFrame(data_pasien_baru)

# Pastikan urutan kolom sesuai
fitur_semua = fitur_numerik + fitur_kategorik
df_pasien = df_pasien[fitur_semua]

# Prediksi probabilitas dan kelas menggunakan model yang telah di-training
proba = model_terbaik.predict_proba(df_pasien)[0]
pred = model_terbaik.predict(df_pasien)[0]

map_keparahan = {0: 'Ringan', 1: 'Sedang', 2: 'Berat'}

print("--- Hasil Prediksi Pasien Baru ---")
print(f"Model yang digunakan: {nama_model_terbaik}")
print(f"Prediksi Kelas: {map_keparahan[pred]} ({pred})")
print(f"Probabilitas Ringan : {proba[0]*100:.2f}%")
print(f"Probabilitas Sedang : {proba[1]*100:.2f}%")
print(f"Probabilitas Berat  : {proba[2]*100:.2f}%")
